# 🌊 MomentumPlane — Interactive Demo

A physics-inspired simulation of **periodic coherent injection** and **discrete quantum hopping** on momentum-space lattices.

This notebook lets you run the full simulation pipeline directly in your browser — no local install needed.

**GitHub:** https://github.com/eninem123/MomentumPlane

## 1. Setup

Clone the repo and install dependencies.

In [ ]:
!git clone https://github.com/eninem123/MomentumPlane.git
%cd MomentumPlane
!pip install -q numpy scipy matplotlib pillow jax jaxlib

## 2. Run a Basic Simulation

Configure the physics parameters and run the full pipeline: **Inject → Quantum Walk → FFT → Momentum Plane**.

In [ ]:
import sys
sys.path.insert(0, '.')

from momentum_plane import MomentumPlanePipeline, SimulationConfig

cfg = SimulationConfig(
    grid_size=96, stride=10, wavevector=(0.5, 0.5), sigma=2.5,
    n_steps=40, coin_type='hadamard', coin_angle=0.0, boundary='periodic',
    record_every=2, seed=42, backend='numpy',
)

result = MomentumPlanePipeline(cfg).run()
print(f"Steps recorded: {len(result['position_history'])}")
print(f"Final momentum peaks: {result['n_peaks'][-1]}")
print(f"Peak intensity: {result['peak_intensities'][-1]:.4f}")

## 3. Visualise the Results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

im0 = axes[0].imshow(result['final_position'], cmap='inferno', origin='lower')
axes[0].set_title('Position Space |ψ|²')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

mom = result['final_momentum_intensity']
im1 = axes[1].imshow(np.log10(mom + 1e-10), cmap='viridis', origin='lower')
axes[1].set_title('Momentum Plane (log intensity)')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

axes[2].plot(result['peak_intensities'], 'o-', color='#e74c3c', linewidth=2)
axes[2].set_title('Peak Intensity Convergence')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Animate the Evolution

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

def update(frame):
    ax1.clear(); ax2.clear()
    ax1.imshow(result['position_history'][frame], cmap='inferno', origin='lower')
    ax1.set_title(f'Position (step {(frame+1)*cfg.record_every})')
    mom = result['momentum_history'][frame]
    ax2.imshow(np.log10(mom + 1e-10), cmap='viridis', origin='lower')
    ax2.set_title(f'Momentum (step {(frame+1)*cfg.record_every})')

anim = FuncAnimation(fig, update, frames=len(result['position_history']), interval=200, blit=False)
plt.close()
HTML(anim.to_jshtml())

## 5. Try Different Physics Regimes

In [ ]:
configs = {
    'Classic': dict(wavevector=(0.8, 0.0), stride=12, coin_type='hadamard'),
    'Diagonal': dict(wavevector=(0.5, 0.5), stride=10, coin_type='hadamard'),
    'Grover': dict(wavevector=(0.6, 0.2), stride=10, coin_type='grover'),
    'Chiral': dict(wavevector=(0.7, 0.0), stride=12, coin_type='hadamard', coin_angle=0.8),
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (name, overrides) in zip(axes, configs.items()):
    c = SimulationConfig(grid_size=64, n_steps=25, record_every=25, seed=42, **overrides)
    r = MomentumPlanePipeline(c).run()
    ax.imshow(np.log10(r['final_momentum_intensity'] + 1e-10), cmap='viridis', origin='lower')
    ax.set_title(f'{name} ({r["n_peaks"][-1]} peaks)')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. JAX Backend (GPU)

Switch to GPU runtime (Runtime → Change runtime type → GPU) and set `backend='jax'` for 10-100x speedup on large grids.

In [ ]:
import jax
print(f'JAX devices: {jax.devices()}')
# cfg_jax = SimulationConfig(grid_size=128, n_steps=50, backend='jax', seed=42)
# result_jax = MomentumPlanePipeline(cfg_jax).run()

## 7. What's Going On Physically?

1. **Inject** — Coherent Gaussian wave-packets at regular intervals, same momentum direction (diffraction grating).
2. **Hop** — DTQW: Hadamard/Grover coin + conditional shift. Walker delocalises.
3. **Transform** — 2D FFT reveals momentum plane: comb of sharp peaks.
4. **Converge** — Phases re-align, peaks sharpen/split/form caustics.

**GitHub:** https://github.com/eninem123/MomentumPlane | **Theory:** docs/theory.md